# DATA

## EXTRACCION DE DATOS
Para el desarrollo de este proyecto se recurrió a dos tipos de fuentes. Por un lado, las pistas de audio utilizadas para construir el dataset fueron obtenidas de Free Music Archive (https://freemusicarchive.org), plataforma de distribución musical que ofrece canciones bajo licencias Creative Commons, permitiendo su uso libre en contextos académicos e investigativos sin restricciones de derechos de autor. Las canciones fueron seleccionadas y organizadas manualmente por género musical, garantizando un mínimo de 50 pistas por categoría.

## PROCESAMIENTO Y MODELAMIENTO DE LOS DATOS

### Librerias 
Librosa: es una librería especializada en análisis de audio y música que permite cargar archivos de sonido y extraer características acústicas. Es el núcleo del proceso de extracción de features en este proyecto.

NumPy: es la librería fundamental para computación numérica en Python. Proporciona estructuras de datos tipo array de alto rendimiento y operaciones matemáticas vectorizadas, siendo la base sobre la que trabajan la mayoría de librerías científicas incluyendo Librosa.

Pandas es la librería estándar para manipulación y análisis de datos tabulares en Python. Permite construir, limpiar y transformar dataframes, que es la estructura utilizada para almacenar y exportar el dataset final en formato CSV.

Os es un módulo nativo de Python que permite interactuar con el sistema operativo, específicamente con el sistema de archivos. En este proyecto se utiliza para recorrer las carpetas de géneros, listar los archivos de audio disponibles y construir las rutas de acceso a cada canción.

### Extracción de Features 
El código implementa un pipeline de extracción automática de características acústicas a partir de archivos de audio organizados por género musical. Para cada canción se carga la pista completa y se extraen tres segmentos de 30 segundos correspondientes al inicio, la mitad y el final de la pista. De cada segmento se calculan 27 features: siete características acústicas globales (tempo, spectral centroid, spectral bandwidth, rolloff, zero crossing rate, RMS energy y chroma STFT) y 20 coeficientes MFCC que representan el timbre de la señal. Cada segmento procesado queda registrado como una fila independiente en el dataset, junto con su etiqueta de género y el nombre del archivo de origen. El resultado final es un archivo CSV estructurado y listo para la etapa de modelado.

In [3]:
import librosa
import numpy as np
import pandas as pd
import os
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

def extraerfeatures(y, sr):
    # Aumentamos a 13 MFCCs para mayor detalle tímbrico
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    
    features = {
        'tempo':               librosa.beat.tempo(y=y, sr=sr)[0],
        'spectral_centroid':   librosa.feature.spectral_centroid(y=y, sr=sr).mean(),
        'spectral_bandwidth':  librosa.feature.spectral_bandwidth(y=y, sr=sr).mean(),
        'rolloff':             librosa.feature.spectral_rolloff(y=y, sr=sr).mean(),
        'zero_crossing_rate':  librosa.feature.zero_crossing_rate(y).mean(),
        'rms':                 librosa.feature.rms(y=y).mean(),
        'chroma_stft':         chroma.mean(),
        'onset_strength_mean': onset_env.mean(),
        'onset_strength_std':  onset_env.std(),
        'pitch_variance':      chroma.var(),
        'spectral_flux_std':   onset_env.std() 
    }
    
    for i, coef in enumerate(mfccs):
        features[f'mfcc{i+1}_mean'] = coef.mean()
        features[f'mfcc{i+1}_std']  = coef.std()
        
    return features

def obtenersegmentos(duracion_total):
    inicio = 0
    mitad  = int(duracion_total / 2) - 15
    final  = int(duracion_total) - 30
    return {'inicio': inicio, 'mitad': mitad, 'final': final}

filas = []
carpeta_raiz = 'data/raw/songs'  # Ruta ajustada al árbol de carpetas actual

generos = [d for d in os.listdir(carpeta_raiz) if os.path.isdir(os.path.join(carpeta_raiz, d))]

for genero in generos:
    carpeta_genero = os.path.join(carpeta_raiz, genero)
    archivos_genero = [f for f in os.listdir(carpeta_genero) if f.endswith(('.mp3', '.wav'))]
    
    for archivo in tqdm(archivos_genero, desc=f"Procesando {genero}"):
        ruta = os.path.join(carpeta_genero, archivo)
        try:
            y, sr = librosa.load(ruta, sr=None)
            duracion_total = librosa.get_duration(y=y, sr=sr)

            if duracion_total < 90:
                continue

            segmentos = obtenersegmentos(duracion_total)

            for nombre_seg, seg_inicio in segmentos.items():
                inicio_puntos = int(seg_inicio * sr)
                fin_puntos    = int((seg_inicio + 30) * sr)
                segmento      = y[inicio_puntos:fin_puntos]

                if len(segmento) >= (30 * sr):
                    features = extraerfeatures(segmento, sr)
                    
                    # COLUMNAS CLAVE PARA EL MODELADO
                    features['label'] = genero      # Género (Target)
                    features['song_id'] = archivo    # Identificador único por canción
                    features['segment_type'] = nombre_seg
                    
                    filas.append(features)
        except Exception:
            continue

df = pd.DataFrame(filas)
df.to_csv('dataset_musical_7_generos.csv', index=False)
print(f"\nDataset creado: {df.shape[0]} filas con columna 'label' y 'song_id'.")

Procesando Vallenato:   0%|          | 0/50 [00:00<?, ?it/s]

Procesando pop:  26%|██▌       | 13/50 [00:27<01:16,  2.07s/it][src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?
[src/libmpg123/id3.c:process_extra():684] error: No extra frame text / valid description?
[src/libmpg123/id3.c:process_extra():684] error: No extra frame text / valid description?
[src/libmpg123/id3.c:process_extra():684] error: No extra frame text / valid description?
[src/libmpg123/id3.c:process_extra():684] error: No extra frame text / valid description?
[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?
Procesando pop:  32%|███▏      | 16/50 [00:33<01:12,  2.14s/it][src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?
[src/libmpg123/id3.c:process_extra():684] error: No extra frame text / valid description?
[src/libmpg123/id3.c:process_extra():684] error: No extra frame text / valid description?
[src/libmpg123/id3.c:process_extra():684] error: No extra frame text /


Dataset creado: 825 filas con columna 'label' y 'song_id'.


## RESULTADOS

In [7]:
df.head(3)

,tempo,spectral_centroid,spectral_bandwidth,rolloff,zero_crossing_rate,rms,chroma_stft,onset_strength_mean,onset_strength_std,pitch_variance,...,mfcc10_std,mfcc11_mean,mfcc11_std,mfcc12_mean,mfcc12_std,mfcc13_mean,mfcc13_std,label,song_id,segment_type
0,140.625000,3155.401300,3505.616624,6743.451164,0.059765,0.176762,0.401536,1.397984,1.720091,0.095407,...,11.798267,-5.197709,11.800679,-4.573185,10.176497,1.182906,9.940129,Vallenato,Jose Carrascal - Como Duele El Frio.mp3,inicio
1,144.230769,3213.328268,3507.036509,6884.226138,0.071339,0.207555,0.317630,1.413439,1.547582,0.089780,...,10.543764,-4.728733,11.399025,-3.438390,9.680881,4.272985,10.252098,Vallenato,Jose Carrascal - Como Duele El Frio.mp3,mitad
2,140.625000,3506.135640,3881.860700,8020.240846,0.064768,0.120561,0.457444,1.188929,1.534752,0.093239,...,9.591031,-4.801034,9.903723,2.310814,7.824638,3.756548,9.018738,Vallenato,Jose Carrascal - Como Duele El Frio.mp3,final


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 825 entries, 0 to 824
Data columns (total 40 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   tempo                825 non-null    float64
 1   spectral_centroid    825 non-null    float64
 2   spectral_bandwidth   825 non-null    float64
 3   rolloff              825 non-null    float64
 4   zero_crossing_rate   825 non-null    float64
 5   rms                  825 non-null    float32
 6   chroma_stft          825 non-null    float32
 7   onset_strength_mean  825 non-null    float32
 8   onset_strength_std   825 non-null    float32
 9   pitch_variance       825 non-null    float32
 10  spectral_flux_std    825 non-null    float32
 11  mfcc1_mean           825 non-null    float32
 12  mfcc1_std            825 non-null    float32
 13  mfcc2_mean           825 non-null    float32
 14  mfcc2_std            825 non-null    float32
 15  mfcc3_mean           825 non-null    float32
 16  m

In [8]:
df.describe()

,tempo,spectral_centroid,spectral_bandwidth,rolloff,zero_crossing_rate,rms,chroma_stft,onset_strength_mean,onset_strength_std,pitch_variance,...,mfcc9_mean,mfcc9_std,mfcc10_mean,mfcc10_std,mfcc11_mean,mfcc11_std,mfcc12_mean,mfcc12_std,mfcc13_mean,mfcc13_std
count,825.000000,825.000000,825.000000,825.000000,825.000000,825.000000,825.000000,825.000000,825.000000,825.000000,...,825.000000,825.000000,825.000000,825.000000,825.000000,825.000000,825.000000,825.000000,825.000000,825.000000
mean,125.035962,2361.187009,2743.308804,4730.838150,0.054239,0.140783,0.415156,0.987743,0.968293,0.086701,...,-3.200731,9.844301,-0.278063,9.355836,-2.737139,9.042941,-0.434714,8.703912,-1.161660,8.535522
std,16.116640,1024.942862,887.608432,2250.703165,0.033496,0.082787,0.114838,0.246979,0.515934,0.012832,...,8.435297,2.786080,8.839309,2.676658,6.639748,2.520716,7.008631,2.616113,5.951339,2.490017
min,71.202532,152.476214,380.693772,203.915434,0.003438,0.001516,0.146911,0.219672,0.174061,0.032094,...,-30.063372,4.130637,-31.061556,4.230135,-33.299133,4.439471,-24.334005,3.767380,-23.209993,3.870547
25%,117.187500,1637.230677,2056.234452,3065.171747,0.033414,0.075604,0.347570,0.816458,0.557344,0.080335,...,-8.710527,7.956171,-6.108758,7.650252,-6.180450,7.318845,-4.209652,6.920586,-4.714614,6.838006
50%,125.000000,2244.525802,2726.900796,4418.014952,0.048466,0.131021,0.422136,1.007301,0.918749,0.086925,...,-2.946333,9.387104,-0.004398,8.964796,-2.347582,8.714445,-0.188023,8.212243,-1.124139,8.086319
75%,133.928571,2912.903845,3330.856717,6174.627288,0.065438,0.189166,0.491079,1.168810,1.279271,0.093982,...,2.017112,11.204125,5.451677,10.500648,1.451100,10.201070,3.841231,9.887187,2.884990,9.698793
max,181.451613,9852.458200,6090.424256,13460.618605,0.450585,0.452624,0.817834,1.633782,3.527388,0.135301,...,24.653418,25.635481,25.956970,26.398521,19.731634,24.607998,42.044216,23.813414,19.481466,22.088600


## Visualización descriptiva por género
A continuación se generan gráficos descriptivos de las características acústicas promedio para cada género musical. Estos gráficos ayudan a comparar el comportamiento de cada género en el dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid', palette='muted')

# Selección de características descriptivas
features = ['tempo', 'spectral_centroid', 'spectral_bandwidth', 'rms', 'chroma_stft']
genero_promedios = df.groupby('label')[features].mean()

fig, axes = plt.subplots(len(genero_promedios), 1, figsize=(12, 4 * len(genero_promedios)), sharex=True)
for ax, (genero, valores) in zip(axes, genero_promedios.iterrows()):
    valores.plot(kind='bar', ax=ax)
    ax.set_title(f'Características promedio - {genero}')
    ax.set_ylabel('Valor promedio')
    ax.set_xlabel('')
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Comparación general de indicadores clave por género
promedios_largos = genero_promedios.reset_index().melt(id_vars='label', var_name='feature', value_name='mean')

plt.figure(figsize=(14, 6))
sns.barplot(data=promedios_largos, x='label', y='mean', hue='feature')
plt.title('Promedio de características acústicas por género')
plt.xlabel('Género')
plt.ylabel('Valor promedio')
plt.xticks(rotation=45)
plt.legend(title='Feature', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

Tempo
Mide los BPM (pulsaciones por minuto) de la canción. Librosa detecta los picos de energía que se repiten en el tiempo y calcula cada cuánto ocurren. Un género como el metal o el disco tiene tempo alto, el blues o el classical lo tiene bajo.

Spectral Centroid
Es el centro de gravedad del espectro de frecuencias. Se calcula como el promedio ponderado de todas las frecuencias presentes en la señal. Un valor alto significa que la energía está concentrada en frecuencias agudas (sonido brillante), uno bajo en frecuencias graves (sonido oscuro y profundo).

Spectral Bandwidth
Mide qué tan dispersas están las frecuencias alrededor del centroid. Si el ancho es grande, la canción tiene mucho contenido en un rango amplio de frecuencias. Si es pequeño, el sonido es más puro y concentrado. Géneros ruidosos como el metal tienen bandwidth alto.

Rolloff
Es la frecuencia por debajo de la cual se encuentra el 85% de la energía total de la señal. Sirve para detectar si una canción tiene mucho contenido en frecuencias altas o bajas. Distingue bien géneros con mucho agudo (rock, metal) de géneros más cálidos (jazz, blues).

Zero Crossing Rate
Cuenta cuántas veces por segundo la señal de audio pasa de un valor positivo a uno negativo. Una señal percusiva o ruidosa cruza el cero muy frecuentemente. Una señal tonal y melódica lo hace pocas veces. Es útil para separar géneros como hiphop y metal de classical y jazz.

RMS (Root Mean Square Energy)
Es la raíz cuadrada del promedio de los cuadrados de las amplitudes de la señal. Representa el volumen real promedio de la canción. Géneros intensos como metal tienen RMS alto, géneros acústicos o suaves como classical lo tienen bajo.

Chroma STFT
Mapea todas las frecuencias de la señal a las 12 notas musicales de la escala cromática (do, do#, re, re#...) y mide cuánto aparece cada una. Captura la armonía y la tonalidad de la canción. Es especialmente útil para distinguir géneros que tienen estructuras armónicas características, como el jazz o el reggae.

onset_strength_mean
Es el promedio de la fuerza con la que ocurren los golpes rítmicos a lo largo del segmento. Un valor alto indica que los beats son intensos y pronunciados, como en el hiphop o el metal. Un valor bajo indica golpes suaves o difusos, como en el classical.

onset_strength_std
Es la desviación estándar de esa fuerza. Mide qué tan regular o irregular es el ritmo. Un valor bajo significa que los beats ocurren con intensidad constante (ritmo mecánico y predecible como el disco o el hiphop). Un valor alto significa que la intensidad varía mucho entre beats (ritmo irregular y expresivo como el jazz).

pitch_variance
Es la varianza del contenido armónico de la canción calculada sobre el histograma de chroma. Mide qué tan estable es la armonía. Un valor bajo indica que la canción se mantiene en pocas notas o tonalidades (reggae, pop). Un valor alto indica que la canción transita por muchas notas distintas con frecuencia (jazz, classical).

mfccX_mean
El promedio del coeficiente MFCC número X a lo largo del segmento. Describe el timbre promedio de la canción en esa dimensión espectral. El coeficiente 1 captura la energía general, los siguientes capturan detalles progresivamente más finos del timbre.

mfccX_std
La desviación estándar del mismo coeficiente. Mide qué tan variable fue ese aspecto del timbre durante el segmento. Un valor alto significa que el sonido cambia mucho en esa dimensión, uno bajo que se mantiene estable. Esta columna es nueva respecto al código original y aporta información sobre la dinámica interna del segmento que la media sola no captura.

label
Indica el genero de la cancion

cancion
Indica el nombre del archivo

Segmento
Indica cual de los 3 segmentos representa la fila